In [1]:
## created a simple RAG project

## steps involved

## LLM MODEL SETUP
## defined the LLM model which we are using (gemini-2.5-flash) along with API key

## DATA INGESTION PIPLINE
## taking the input text
## splitting it into the chunks
## converting those chunks into the vectors (using huggingface - all-MiniLM-L6-v2 embedding model)
## storing those vectors into the vector db (used chromadb which stores vectors into local staorage)

## RETRIEVAL PIPELINE
## crated a retriever which retrieves relevant chunks (texts)
## created a prompt template in which retrieved chunks and a user question is included (this template would be provided to LLM model to generate the answer)
## created a helper function (format_docs) which combines all the extracted chunks (texts) into a single piece of text (called as context)

## WORKFLOW BUILDING & EXECUTION
## created a automated workflow using Langchain Expression Language
## actual execution of this workflow by entering the user question

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/Users/ashish/langchain codes/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ashish/langchain codes/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# embedding model

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12960.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# llm model

import shutil
import os

# Delete the old database folder if it exists (to avoid duplication of vectors)
if os.path.exists("./gemini_rag_db"):
    shutil.rmtree("./gemini_rag_db")


os.environ["GOOGLE_API_KEY"] = "put your api key here"
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

In [5]:
raw_text = """
The Artemis 2 mission was the first crewed flight in NASA’s Artemis program, representing a pivotal bridge between the uncrewed tests of the past and the upcoming lunar landings. Launched on April 1, 2026, the mission carried four astronauts on a 10-day journey that circled the Moon and returned safely to Earth on April 10. This flight was the first time humans had traveled beyond low Earth orbit since the Apollo 17 mission in 1972, serving as a comprehensive "shakedown cruise" to ensure that the Space Launch System (SLS) rocket and the Orion spacecraft were ready for the rigors of deep-space human travel.

During the mission, the crew conducted a high-altitude "lunar flyby" using a free-return trajectory. This path allowed the spacecraft to loop around the far side of the Moon, using lunar gravity as a "slingshot" to pull them back toward Earth without needing a massive engine burn to turn around. At their furthest point, the crew reached a distance of 252,756 miles from Earth, breaking the all-time record for the farthest humans have ever traveled into space. While they did not land on the surface, the astronauts performed intensive manual flight maneuvers and tested essential life-support systems, such as air scrubbers and water recycling, which were not fully active during previous uncrewed flights.

Beyond the technical milestones, Artemis 2 was a historic moment for international and social representation in space. The crew—Commander Reid Wiseman, Pilot Victor Glover, and Mission Specialists Christina Koch and Jeremy Hansen—included the first woman, the first person of color, and the first non-American to venture to the Moon's vicinity. Their successful splashdown in the Pacific Ocean has officially cleared the way for Artemis 3, which aims to land astronauts near the lunar South Pole in late 2027 or 2028.
"""

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_text(raw_text)
chunks

['The Artemis 2 mission was the first crewed flight in NASA’s Artemis program, representing a pivotal',
 'a pivotal bridge between the uncrewed tests of the past and the upcoming lunar landings. Launched',
 'landings. Launched on April 1, 2026, the mission carried four astronauts on a 10-day journey that',
 '10-day journey that circled the Moon and returned safely to Earth on April 10. This flight was the',
 'This flight was the first time humans had traveled beyond low Earth orbit since the Apollo 17',
 'since the Apollo 17 mission in 1972, serving as a comprehensive "shakedown cruise" to ensure that',
 'to ensure that the Space Launch System (SLS) rocket and the Orion spacecraft were ready for the',
 'were ready for the rigors of deep-space human travel.',
 'During the mission, the crew conducted a high-altitude "lunar flyby" using a free-return',
 'using a free-return trajectory. This path allowed the spacecraft to loop around the far side of the',
 'the far side of the Moon, using 

In [7]:
print(type(chunks))
print(len(chunks))

<class 'list'>
24


In [8]:
vector_db = Chroma.from_texts(
    texts=chunks, 
    embedding=embeddings, 
    persist_directory="./gemini_rag_db"
)

vector_db

In [9]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x1282e4c20>, search_kwargs={'k': 3})

In [14]:
template = """
You are a story teller. Use the following context to answer the question.

Context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

In [15]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

test_query = "What is the topic about?"
found_docs = retriever.invoke(test_query)

print(found_docs)
print(format_docs(found_docs))

[Document(id='9783488b-de02-47a0-aa5e-6960e24f667f', metadata={}, page_content='and social representation in space. The crew—Commander Reid Wiseman, Pilot Victor Glover, and'), Document(id='480dd7f0-df8a-4ebb-9d35-e1090d0da4ce', metadata={}, page_content='were ready for the rigors of deep-space human travel.'), Document(id='f6047699-4e8e-4400-8989-5db78ec06227', metadata={}, page_content='a pivotal bridge between the uncrewed tests of the past and the upcoming lunar landings. Launched')]
and social representation in space. The crew—Commander Reid Wiseman, Pilot Victor Glover, and

were ready for the rigors of deep-space human travel.

a pivotal bridge between the uncrewed tests of the past and the upcoming lunar landings. Launched


In [16]:
# This function combines the retrieved chunks into one string for Gemini
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The RAG Workflow
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
# 6. EXECUTION
query = "when this mission is carried out"
response = rag_chain.invoke(query)

print(f"\n--- RAG Response ---\n{response}")


--- RAG Response ---
This mission was carried out on **April 1, 2026**.
